In [51]:
import numpy as np

In [53]:
import os

In [55]:
import cv2

In [57]:
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

In [59]:
dataset_path = "D:/Photos"
images, labels = [], []
label_map = {}

In [61]:
label_id = 0
for person_name in os.listdir(dataset_path):
    person_path = os.path.join(dataset_path, person_name)
    if not os.path.isdir(person_path):
        continue

    label_map[label_id] = person_name
    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        faces = face_cascade.detectMultiScale(img, 1.1, 5)
        for (x, y, w, h) in faces:
            face_roi = img[y:y+h, x:x+w]
            images.append(face_roi)
            labels.append(label_id)
    label_id += 1

images = np.array(images, dtype="object")
labels = np.array(labels)

In [62]:
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.train(images, labels)

AttributeError: module 'cv2.face' has no attribute 'LBPHFaceRecognizer_create'

In [ ]:
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5)

    for (x, y, w, h) in faces:
        face_roi = gray[y:y+h, x:x+w]
        label, confidence = recognizer.predict(face_roi)
        name = label_map[label]
        cv2.putText(frame, f"{name} ({int(confidence)})", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)
        cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)

    cv2.imshow("Face Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

In [ ]:
cap.release()
cv2.destroyAllWindows()